# 05 · Chest X-ray CNN

> **Synthetic data only.** Every dataset used in this notebook is synthetic (Synthea, seed 42) or research-use-only (NIH ChestX-ray14). There is no PHI. None of these models is clinically validated or cleared for patient care.

Outline the 14-label chest X-ray classifier (DenseNet121) trained on the **research-use-only** NIH ChestX-ray14 dataset. We parse the label CSV schema, build the multi-hot target, and sketch the model head. No images are shipped; cells are illustrative.

> **License reminder.** NIH ChestX-ray14 is *research use only* and noisy (labels NLP-mined from reports). This repo ships no images; the training job reads a local mirror at `CHESTXRAY_DIR`.

## The 14 finding labels
Canonical order matching the serving `NIH_LABELS` and the training job.

In [ ]:
from __future__ import annotations
import numpy as np, pandas as pd
NIH_LABELS = (
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
    'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia',
)
len(NIH_LABELS)

## Parse `Data_Entry_2017.csv`
The `Finding Labels` column is pipe-delimited; we map it to a 14-dim multi-hot vector. Here we use a tiny synthetic stand-in row set.

In [ ]:
raw = pd.DataFrame({
    'Image Index': ['0001.png', '0002.png', '0003.png'],
    'Finding Labels': ['Cardiomegaly|Effusion', 'No Finding', 'Pneumonia'],
    'Patient ID': [1, 2, 3],
})
def multihot(labels: str) -> list[int]:
    present = set(labels.split('|'))
    return [int(l in present) for l in NIH_LABELS]
Y = np.array([multihot(s) for s in raw['Finding Labels']])
pd.DataFrame(Y, columns=list(NIH_LABELS))

## Per-label positive weights
`BCEWithLogitsLoss` uses `pos_weight` from label frequency to counter class imbalance.

In [ ]:
freq = Y.mean(axis=0)
pos_weight = np.where(freq > 0, (1 - freq) / np.maximum(freq, 1e-6), 1.0)
pd.Series(np.round(pos_weight, 2), index=NIH_LABELS).head()

## Model head sketch
DenseNet121 backbone (ImageNet) with the classifier replaced by a 14-unit linear head. Imported lazily so the notebook loads without torchvision weights.

In [ ]:
try:
    import torch
    from torchvision import models
    net = models.densenet121(weights=None)
    in_features = net.classifier.in_features
    net.classifier = torch.nn.Linear(in_features, len(NIH_LABELS))
    print('head out features:', net.classifier.out_features)
except Exception as exc:  # torchvision optional in EDA env
    print('torchvision not available here:', type(exc).__name__)

## Evaluation: per-label AUROC
The job reports patient-disjoint per-label AUROC. We demo the metric on synthetic scores.

In [ ]:
from medflow_ml.evaluation.metrics import auroc
rng = np.random.default_rng(42)
y_true = rng.integers(0, 2, size=200)
y_score = 0.5 * y_true + 0.5 * rng.random(200)
print('demo per-label AUROC:', round(auroc(y_true, y_score), 3))

### Notes
* Production: `make download-chestxray` then `python -m medflow_ml.jobs.train_xray` registers `chest-xray-14`.
* Grad-CAM overlays are produced at serving time.
* Next: clinical NLP (notebook 06).